# LeetCode 3225 — Maximum Score From Grid Operations bằng Python

- Link: https://leetcode.com/problems/maximum-score-from-grid-operations/
- Độ khó: **Hard**
- Chủ đề: Quy về chọn **chiều cao tô đen** `h[j] ∈ [0, n]` cho mỗi cột; điểm của cột `j` chỉ phụ thuộc `(h[j−1], h[j], h[j+1])` ⇒ **DP trượt cửa sổ 2 cột** + tổng tiền tố theo cột + tối ưu chuyển trạng thái bằng **prefix/suffix max**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumScore(self, grid: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra mọi chuỗi thao tác chỉ để lại cho mỗi cột một **chiều cao đen** `h[j]` (các ô hàng `< h[j]` đen) — thứ tự và số thao tác không quan trọng.
2. Viết điểm của cột `j`: các ô trắng `i ≥ h[j]` có hàng xóm ngang đen, tức `i < max(h[j−1], h[j+1])` ⇒ điểm `= S_j(max(h[j−1], h[j+1])) − S_j(h[j])` nếu dương, với `S_j` là tổng tiền tố của cột `j`.
3. Thiết kế DP trạng thái `(h[j−1], h[j])`, chuyển sang `h[j+1]`: `O(n⁴)`.
4. Tách trường hợp `max(a, c) = a` hay `= c` để dùng **suffix max / prefix max** ⇒ `O(n³) ≈ 10⁶` với `n = 100`.
5. Hiểu thêm lời giải "pick / skip" — mỗi cột chỉ nhận điểm từ **một phía**.

## 2. Đề bài & ví dụ

Cho ma trận `n × n` `grid`, ban đầu mọi ô **trắng**. Một thao tác chọn ô `(i, j)` và tô đen **mọi ô của cột `j` từ hàng 0 xuống hàng `i`**.

**Điểm** của lưới là tổng `grid[i][j]` của mọi ô `(i, j)` **trắng** và có **ít nhất một ô kề ngang** (trái hoặc phải) **đen**. Trả về điểm lớn nhất đạt được sau một số thao tác bất kỳ.

**Ví dụ 1:** `grid = [[0,0,0,0,0],[0,0,3,0,0],[0,1,0,0,0],[5,0,0,3,0],[0,0,0,0,2]]` → `11` (tô cột 1 tới hàng 3, cột 4 tới hàng cuối: `5 + 3 + 3`).

**Ví dụ 2:** `grid = [[10,9,0,0,15],[7,1,0,8,0],[5,20,0,11,0],[0,0,0,1,2],[8,12,1,10,3]]` → `94`.

**Ràng buộc:** `1 ≤ n ≤ 100`; `0 ≤ grid[i][j] ≤ 10^9`.

In [1]:
from typing import List
import itertools

NEG = float('-inf')
examples = [
    ([[0, 0, 0, 0, 0], [0, 0, 3, 0, 0], [0, 1, 0, 0, 0], [5, 0, 0, 3, 0], [0, 0, 0, 0, 2]], 11),
    ([[10, 9, 0, 0, 15], [7, 1, 0, 8, 0], [5, 20, 0, 11, 0], [0, 0, 0, 1, 2], [8, 12, 1, 10, 3]], 94),
]
for g, exp in examples:
    print(len(g), 'x', len(g), '-> kỳ vọng', exp)

5 x 5 -> kỳ vọng 11
5 x 5 -> kỳ vọng 94


## 3. Kiến thức nền

### 3.1. Mô hình chiều cao

Tô đen cột `j` tới hàng `i` = "cột `j` đen ở mọi hàng `≤ i`". Nhiều thao tác trên cùng cột chỉ giữ thao tác sâu nhất. Vậy trạng thái cuối mô tả bởi `h[j] ∈ {0, 1, …, n}` = **số ô đen** đầu cột `j` (`h[j] = 0`: không thao tác).

Ô `(i, j)` trắng ⇔ `i ≥ h[j]`; có hàng xóm trái đen ⇔ `j > 0` và `i < h[j−1]`; phải đen ⇔ `j < n−1` và `i < h[j+1]`.

In [2]:
def score_of_heights(grid, h):
    n = len(grid); total = 0
    for j in range(n):
        for i in range(n):
            if i >= h[j] and ((j > 0 and h[j - 1] > i) or (j + 1 < n and h[j + 1] > i)):
                total += grid[i][j]
    return total

g1 = examples[0][0]
print('ví dụ 1 với h = [0,4,0,0,5]:', score_of_heights(g1, [0, 4, 0, 0, 5]))

ví dụ 1 với h = [0,4,0,0,5]: 11


### 3.2. Điểm của một cột qua tổng tiền tố

Đặt `S_j(x) = grid[0][j] + … + grid[x−1][j]`. Các ô được tính của cột `j` là hàng `i` với `h[j] ≤ i < max(L, R)` (với `L = h[j−1]`, `R = h[j+1]`, cột ngoài biên coi chiều cao 0):

```
gain_j(L, h, R) = S_j(max(L, R)) − S_j(h)   nếu max(L, R) > h, ngược lại 0
```

Tổng điểm = `Σ_j gain_j(h[j−1], h[j], h[j+1])`.

In [3]:
def col_prefix(grid):
    n = len(grid)
    S = [[0] * (n + 1) for _ in range(n)]
    for j in range(n):
        for i in range(n):
            S[j][i + 1] = S[j][i] + grid[i][j]
    return S

def score_by_gain(grid, h):
    n = len(grid); S = col_prefix(grid); total = 0
    for j in range(n):
        L = h[j - 1] if j > 0 else 0; R = h[j + 1] if j + 1 < n else 0
        x = max(L, R)
        total += S[j][x] - S[j][h[j]] if x > h[j] else 0
    return total

import random
rng = random.Random(3225)
for _ in range(300):
    n = rng.randint(1, 5); g = [[rng.randint(0, 9) for _ in range(n)] for _ in range(n)]
    h = [rng.randint(0, n) for _ in range(n)]
    assert score_of_heights(g, h) == score_by_gain(g, h)
print('Công thức gain theo tổng tiền tố khớp với đếm trực tiếp trên 300 cấu hình.')

Công thức gain theo tổng tiền tố khớp với đếm trực tiếp trên 300 cấu hình.


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

- Điểm cột `j` chỉ "nhìn" 3 cột liên tiếp ⇒ đi từ trái sang phải, nhớ **hai** chiều cao gần nhất `(h[j−1], h[j])` là đủ: khi chọn `h[j+1]` ta tính xong cột `j` mãi mãi.
- Trực giác về hình dạng: một cột đen cao làm "lộ" ô trắng của **hai** cột bên cạnh; nhưng tự nó mất các ô bị đen. Cân bằng này không tham lam được (mục 13, lỗi 3).
- Ví dụ 1: chọn `h = [0, 4, 0, 0, 5]`: cột 0 lộ hàng 0..3 (được `5`), cột 2 lộ hàng 0..3 (được `3`), cột 3 lộ hàng 0..4 nhờ cột 4 (được `3`) ⇒ `11`.

## 5. Cách 1 — Brute Force: thử mọi vector chiều cao

`(n+1)^n` vector — chỉ dùng cho `n ≤ 4` làm oracle.

In [4]:
def brute(grid):
    n = len(grid)
    return max(score_of_heights(grid, h) for h in itertools.product(range(n + 1), repeat=n))

for g, exp in examples:
    print('brute:', brute(g), '| kỳ vọng', exp)

brute: 11 | kỳ vọng 11
brute: 94 | kỳ vọng 94


## 6. Ý tưởng thuật toán / bước đột phá

DP: `dp[a][b]` = điểm tốt nhất của các cột `< j` khi `h[j−1] = a`, `h[j] = b` (khởi đầu `j = 0`: `a = 0` là cột ảo). Chọn `c = h[j+1]` (với cột cuối, `c = 0` ảo):

```
new[b][c] = max_a  dp[a][b] + gain_j(max(a, c), b)
```

`gain_j` chỉ phụ thuộc `max(a, c)`. Tách:

- `a ≥ c`: số hạng `dp[a][b] + g(a, b)` **không phụ thuộc** `c` ⇒ lấy **suffix max** theo `a` từ `c` trở lên.
- `a < c`: số hạng `dp[a][b] + g(c, b)` ⇒ `(prefix max của dp[a][b] với a < c) + g(c, b)`.

Mỗi `(j, b)` tốn `O(n)` ⇒ tổng `O(n³)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP trạng thái hai cột, chuyển trực tiếp `O(n⁴)`

In [5]:
def dp_n4(grid):
    n = len(grid)
    if n == 1:
        return 0
    S = col_prefix(grid)
    dp = {(0, b): 0 for b in range(n + 1)}
    for j in range(n):
        cs = range(n + 1) if j < n - 1 else [0]
        new = {}
        for (a, b), v in dp.items():
            for c in cs:
                x = max(a, c)
                w = v + (S[j][x] - S[j][b] if x > b else 0)
                if new.get((b, c), NEG) < w:
                    new[(b, c)] = w
        dp = new
    return max(dp.values())

for g, exp in examples:
    print('O(n^4):', dp_n4(g), '| kỳ vọng', exp)

O(n^4): 11 | kỳ vọng 11
O(n^4): 94 | kỳ vọng 94


Với `n = 100`: `≈ 100 · 101³ ≈ 10⁸` phép — quá chậm cho Python, nhưng là cầu nối rõ ràng từ mô hình tới lời giải.

### 7.2. Cách 3 — DP `O(n³)` với suffix/prefix max (lời giải chính)

In [6]:
def dp_n3(grid):
    n = len(grid)
    if n == 1:
        return 0
    S = col_prefix(grid)
    dp = [[NEG] * (n + 1) for _ in range(n + 1)]
    dp[0] = [0] * (n + 1)                       # h[-1] = 0 (cột ảo), h[0] tuỳ ý
    for j in range(n):
        Sj = S[j]
        new = [[NEG] * (n + 1) for _ in range(n + 1)]
        for b in range(n + 1):
            suf = [NEG] * (n + 2)               # suf[c] = max_{a>=c} dp[a][b] + g(a, b)
            for a in range(n, -1, -1):
                v = dp[a][b] + (Sj[a] - Sj[b] if a > b else 0)
                suf[a] = v if v > suf[a + 1] else suf[a + 1]
            pre = NEG                           # max_{a<c} dp[a][b]
            row = new[b]
            for c in range(n + 1):
                v = suf[c]
                if pre > NEG:
                    w = pre + (Sj[c] - Sj[b] if c > b else 0)
                    if w > v:
                        v = w
                row[c] = v
                if dp[c][b] > pre:
                    pre = dp[c][b]
        dp = new
    return max(dp[b][0] for b in range(n + 1))   # h[n] = 0 (cột ảo)

for g, exp in examples:
    print('O(n^3):', dp_n3(g), '| kỳ vọng', exp)

O(n^3): 11 | kỳ vọng 11
O(n^3): 94 | kỳ vọng 94


### 7.3. Cách 4 — DP "pick / skip": mỗi cột nhận điểm từ **đúng một phía**

Vì `grid ≥ 0`, điểm cột `j` là `max(điểm nhờ trái, điểm nhờ phải)` — tương đương "chọn một phía để tính". Lưu cho mỗi chiều cao `cur` của cột `j`:

- `pick[cur]`: cột `j` **đã** được tính điểm nhờ cột trái (nếu có) — không được tính thêm nhờ cột phải.
- `skip[cur]`: cột `j` **chưa** được tính, để dành cho cột phải.

Chuyển từ cột `j−1` (cao `prev`) sang cột `j` (cao `cur`):

- `cur > prev`: cột `j−1` được lộ đoạn `[prev, cur)` nhờ cột `j` ⇒ chỉ hợp lệ nếu cột `j−1` đang `skip`; cột `j` không được gì từ trái (trái thấp hơn) ⇒ cả `pick` và `skip` của `cur` nhận `skip[prev] + S_{j−1}(cur) − S_{j−1}(prev)`.
- `cur ≤ prev`: cột `j` lộ đoạn `[cur, prev)` nhờ cột `j−1`: `pick[cur] ← pick[prev] + S_j(prev) − S_j(cur)`; hoặc bỏ không tính để dành phải: `skip[cur] ← pick[prev]`.

(Dùng `pick[prev]` ở nhánh thứ hai là đủ vì `pick ≥ skip` từng điểm: `pick` là "đã chốt", luôn có thể coi như chốt 0.)

In [7]:
def pick_skip(grid):
    n = len(grid)
    if n == 1:
        return 0
    S = col_prefix(grid)
    pick = [0] * (n + 1); skip = [0] * (n + 1)
    for j in range(1, n):
        cp = [0] * (n + 1); cs = [0] * (n + 1)
        A = S[j - 1]; B = S[j]
        for cur in range(n + 1):
            bp = bs = 0
            for prev in range(n + 1):
                if cur > prev:
                    v = skip[prev] + A[cur] - A[prev]
                    if v > bp: bp = v
                    if v > bs: bs = v
                else:
                    v = pick[prev] + B[prev] - B[cur]
                    if v > bp: bp = v
                    if pick[prev] > bs: bs = pick[prev]
            cp[cur] = bp; cs[cur] = bs
        pick, skip = cp, cs
    return max(pick)

for g, exp in examples:
    print('pick/skip:', pick_skip(g), '| kỳ vọng', exp)

pick/skip: 11 | kỳ vọng 11
pick/skip: 94 | kỳ vọng 94


## 8. TRACE từng bước

Truy vết DP `O(n⁴)` có lưu cha để **khôi phục** vector chiều cao tối ưu và vẽ lưới (`#` = đen, `*` = ô trắng được tính điểm, `.` = ô trắng không tính).

In [8]:
def dp_with_heights(grid):
    n = len(grid)
    if n == 1:
        return 0, [0]
    S = col_prefix(grid)
    dp = {(0, b): (0, (b,)) for b in range(n + 1)}
    for j in range(n):
        cs = range(n + 1) if j < n - 1 else [0]
        new = {}
        for (a, b), (v, hs) in dp.items():
            for c in cs:
                x = max(a, c)
                w = v + (S[j][x] - S[j][b] if x > b else 0)
                if (b, c) not in new or new[(b, c)][0] < w:
                    new[(b, c)] = (w, hs + (c,))
        dp = new
    v, hs = max(dp.values())
    return v, list(hs[:n])

def draw(grid, h):
    n = len(grid)
    for i in range(n):
        row = ''
        for j in range(n):
            if i < h[j]:
                row += '  #'
            elif (j > 0 and h[j - 1] > i) or (j + 1 < n and h[j + 1] > i):
                row += f'{"*" + str(grid[i][j]):>3s}'
            else:
                row += '  .'
        print(row)

for g, exp in examples:
    v, h = dp_with_heights(g)
    print('điểm', v, '| chiều cao', h, '| kiểm tra', score_of_heights(g, h))
    draw(g, h); print()

điểm 11 | chiều cao [0, 4, 0, 0, 5] | kiểm tra 11
 *0  # *0 *0  #
 *0  # *3 *0  #
 *0  # *0 *0  #
 *5  # *0 *3  #
  .  .  . *0  #

điểm 94 | chiều cao [0, 2, 5, 1, 0] | kiểm tra 94
*10  #  #  #*15
 *7  #  # *8  .
  .*20  #*11  .
  . *0  # *1  .
  .*12  #*10  .



In [9]:
# In bảng DP theo từng cột cho ví dụ 1 (chỉ in giá trị tốt nhất theo h[j])
g = examples[0][0]; n = len(g); S = col_prefix(g)
dp = {(0, b): 0 for b in range(n + 1)}
for j in range(n):
    cs = range(n + 1) if j < n - 1 else [0]
    new = {}
    for (a, b), v in dp.items():
        for c in cs:
            x = max(a, c); w = v + (S[j][x] - S[j][b] if x > b else 0)
            if new.get((b, c), NEG) < w:
                new[(b, c)] = w
    dp = new
    best_by_c = {}
    for (b, c), v in dp.items():
        best_by_c[c] = max(best_by_c.get(c, NEG), v)
    print(f'sau khi chốt cột {j}: best theo h[{j + 1}] =', [best_by_c.get(c) for c in sorted(best_by_c)])

sau khi chốt cột 0: best theo h[1] = [0, 0, 0, 0, 5, 5]
sau khi chốt cột 1: best theo h[2] = [5, 5, 5, 5, 5, 5]
sau khi chốt cột 2: best theo h[3] = [8, 8, 8, 8, 8, 8]
sau khi chốt cột 3: best theo h[4] = [8, 8, 8, 8, 11, 11]
sau khi chốt cột 4: best theo h[5] = [11]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maximumScore(self, grid: List[List[int]]) -> int:
        n = len(grid)
        if n == 1:
            return 0
        S = [[0] * (n + 1) for _ in range(n)]
        for j in range(n):
            col = S[j]
            for i in range(n):
                col[i + 1] = col[i] + grid[i][j]
        NEG = float('-inf')
        dp = [[NEG] * (n + 1) for _ in range(n + 1)]
        dp[0] = [0] * (n + 1)
        for j in range(n):
            Sj = S[j]
            new = [[NEG] * (n + 1) for _ in range(n + 1)]
            for b in range(n + 1):
                sb = Sj[b]
                suf = [NEG] * (n + 2)
                for a in range(n, -1, -1):
                    v = dp[a][b]
                    if a > b:
                        v += Sj[a] - sb
                    suf[a] = v if v > suf[a + 1] else suf[a + 1]
                pre = NEG
                row = new[b]
                for c in range(n + 1):
                    v = suf[c]
                    if pre > NEG:
                        w = pre + (Sj[c] - sb if c > b else 0)
                        if w > v:
                            v = w
                    row[c] = v
                    if dp[c][b] > pre:
                        pre = dp[c][b]
            dp = new
        return max(dp[b][0] for b in range(n + 1))

for g, exp in examples:
    got = Solution().maximumScore(g)
    print(got, 'OK' if got == exp else 'SAI')

11 OK
94 OK


## 10. Vì sao thuật toán đúng

1. **Mô hình đầy đủ:** mọi tập thao tác cho đúng một vector `h`, và mọi vector `h` đạt được (mỗi cột `h[j] > 0` một thao tác tại `(h[j]−1, j)`). Điểm chỉ phụ thuộc `h`.
2. **Phân rã:** tổng điểm `= Σ_j gain_j(h[j−1], h[j], h[j+1])` (mục 3.2, đã kiểm chứng). Mỗi số hạng dùng 3 chiều cao liên tiếp ⇒ DP trên cặp `(h[j−1], h[j])` duyệt đúng mọi `h` và cộng đủ mỗi số hạng **đúng một lần** (khi chọn `h[j+1]`).
3. **Tối ưu `O(n³)`:** với `a ≥ c`, `max(a, c) = a`; với `a < c`, `= c`. Hai miền phủ mọi `a` và rời nhau ⇒ max trên mỗi miền rồi lấy max là đúng `max_a`.
4. **Biên:** cột ảo trái/phải chiều cao 0 không tô đen ô nào và không có điểm. `n = 1`: không có hàng xóm ngang ⇒ `0`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O((n+1)^n · n²)` | `O(n)` |
| 2. DP hai cột trực tiếp | `O(n⁴)` | `O(n²)` |
| 3. DP + suffix/prefix max | `O(n³)` | `O(n²)` |
| 4. Pick / skip | `O(n³)` | `O(n)` mỗi cột (+ `O(n²)` tiền tố) |

Với `n = 100`: `≈ 10⁶` bước cơ bản cho cách 3/4 — dưới 1 giây.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'dp_n4': dp_n4, 'dp_n3': dp_n3, 'pick_skip': pick_skip,
         'submit': lambda g: Solution().maximumScore(g)}
edge = [([[7]], 0), ([[0, 0], [0, 0]], 0), ([[1, 2], [3, 4]], 6), ([[5, 0], [5, 0]], 10),
        ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 6), ([[0, 9, 0], [0, 9, 0], [0, 9, 0]], 27)]
for g, exp in examples + edge:
    if len(g) <= 4:
        assert brute(g) == exp, (g, brute(g))
    for name, f in impls.items():
        assert f(g) == exp, (name, g, f(g))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích vài ca biên: `[[5,0],[5,0]]` — tô đen hết cột 1 (toàn 0) để lộ cả cột 0 ⇒ `10`. `[[0,9,0],…]` — tô hết cột 0 (hoặc 2) để lộ trọn cột giữa ⇒ `27`. Lưới toàn 1 cỡ 3: tô hết cột giữa lộ 2 cột bên ⇒ `6`.

### 12.2. Đối chiếu ngẫu nhiên với brute (`n ≤ 4`)

In [12]:
cnt = 0
for _ in range(300):
    n = rng.randint(1, 4)
    g = [[rng.choice([0, 0, rng.randint(0, 9)]) for _ in range(n)] for _ in range(n)]
    exp = brute(g)
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    cnt += 1
print('Khớp brute trên', cnt, 'lưới ngẫu nhiên.')

Khớp brute trên 300 lưới ngẫu nhiên.


In [13]:
# n vừa: so ba cách DP với nhau
for _ in range(20):
    n = rng.randint(5, 18)
    g = [[rng.randint(0, 100) for _ in range(n)] for _ in range(n)]
    x = dp_n4(g)
    assert x == dp_n3(g) == pick_skip(g) == Solution().maximumScore(g)
print('Ba cách DP khớp nhau với n ≤ 18.')

Ba cách DP khớp nhau với n ≤ 18.


### 12.3. Ca lớn `n = 100`, giá trị tới `10^9`

In [14]:
import time
N = 100
big = {
    'ngẫu nhiên': [[rng.randint(0, 10 ** 9) for _ in range(N)] for _ in range(N)],
    'toàn 10^9': [[10 ** 9] * N for _ in range(N)],
    'cột xen kẽ': [[10 ** 9 if j % 2 else 0 for j in range(N)] for _ in range(N)],
}
for name, g in big.items():
    t = time.time(); x = Solution().maximumScore(g); t1 = time.time() - t
    t = time.time(); y = pick_skip(g); t2 = time.time() - t
    assert x == y
    print(f'{name:12s}: {x:16d} | nộp {t1:.2f}s | pick/skip {t2:.2f}s')
assert Solution().maximumScore(big['cột xen kẽ']) == 50 * N * 10 ** 9
print('Cột xen kẽ: tô đen hết các cột 0 -> lộ trọn 50 cột 10^9: đúng.')

ngẫu nhiên  :    3317604736846 | nộp 0.33s | pick/skip 0.11s


toàn 10^9   :    6600000000000 | nộp 0.23s | pick/skip 0.12s


cột xen kẽ  :    5000000000000 | nộp 0.23s | pick/skip 0.10s


Cột xen kẽ: tô đen hết các cột 0 -> lộ trọn 50 cột 10^9: đúng.


## 13. Các lỗi người mới hay gặp

Các hàm sai dưới đây dùng chung khung DP ba chiều cao `generic_dp(grid, gain, heights)` (mũ, chỉ chạy lưới nhỏ) để thấy rõ **chỗ sai nằm ở công thức**.

In [15]:
def generic_dp(grid, gain, heights=None):
    n = len(grid); S = col_prefix(grid)
    heights = list(heights) if heights is not None else list(range(n + 1))
    best = NEG
    for h in itertools.product(heights, repeat=n):
        t = 0
        for j in range(n):
            L = h[j - 1] if j > 0 else 0; R = h[j + 1] if j + 1 < n else 0
            t += gain(S[j], L, h[j], R)
        best = max(best, t)
    return best

correct_gain = lambda Sj, L, b, R: max(0, Sj[max(L, R)] - Sj[b])
print('khung chung với gain đúng, ví dụ 1:', generic_dp(examples[0][0], correct_gain))

khung chung với gain đúng, ví dụ 1: 11


### Lỗi 1 — Cộng điểm từ trái và từ phải **riêng rẽ** (đếm đôi)

In [16]:
double_gain = lambda Sj, L, b, R: max(0, Sj[L] - Sj[b]) + max(0, Sj[R] - Sj[b])   # SAI

g = [[3, 0, 1], [1, 2, 1], [1, 3, 2]]
print('sai :', generic_dp(g, double_gain), '| đúng:', Solution().maximumScore(g))

sai : 10 | đúng: 9


**Vì sao sai:** ô trắng có **cả hai** hàng xóm đen chỉ được tính **một lần**. Lời giải DP chỉ nhớ một cột trước (`dp[h[j]]`) và "cộng hai chiều" mắc đúng lỗi này — vì thế phải nhớ hai cột, hoặc dùng pick/skip.

### Lỗi 2 — Tính cả các ô đã bị tô đen của chính cột

In [17]:
own_gain = lambda Sj, L, b, R: Sj[max(L, R)]                                    # SAI: quên trừ S_j(h[j])

g = [[2, 1], [3, 0]]
print('sai :', generic_dp(g, own_gain), '| đúng:', Solution().maximumScore(g))

sai : 6 | đúng: 5


**Vì sao sai:** chỉ ô **trắng** mới được tính. Với `[[2,1],[3,0]]`, công thức sai cho tô cả hai cột rồi vẫn tính cột 0.

### Lỗi 3 — Tham lam: tô đen **trọn** các cột chẵn hoặc các cột lẻ

In [18]:
def wrong_alternate(grid):
    n = len(grid); best = 0
    for parity in (0, 1):
        h = [n if j % 2 == parity else 0 for j in range(n)]
        best = max(best, score_of_heights(grid, h))
    return best                                                                    # SAI

g = [[2, 2, 3], [0, 1, 0], [0, 2, 0]]
print('sai :', wrong_alternate(g), '| đúng:', Solution().maximumScore(g))

sai : 5 | đúng: 6


**Vì sao sai:** chiều cao tối ưu thường **lưng chừng** (ở đây tối ưu duy nhất là `h = [3, 1, 0]`: tô trọn cột 0 để lộ hàng 1..2 của cột 1 (`1 + 2`), và cột 1 chỉ tô hàng 0 để lộ ô `3` của cột 2 — tổng `6`; còn mẫu xen kẽ chỉ được `5`). Mẫu xen kẽ toàn phần chỉ đúng trong vài lưới đặc biệt.

### Lỗi 4 — Quên chiều cao 0 (bắt mọi cột phải có thao tác)

In [19]:
g = [[2, 1], [3, 0]]
print('sai :', generic_dp(g, correct_gain, heights=range(1, len(g) + 1)), '| đúng:', Solution().maximumScore(g))

sai : 3 | đúng: 5


**Vì sao sai:** "một số thao tác bất kỳ" gồm cả **không** thao tác trên cột — cột muốn được lộ trọn phải có `h = 0`. Với `[[2,1],[3,0]]`: tối ưu là `h = [0, 2]` lộ cả cột 0 (`5`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tô từ đỉnh xuống" ⇒ mỗi cột một **chiều cao**; trạng thái cuối, không phải chuỗi thao tác.
- Đóng góp phụ thuộc **cửa sổ k cột** ⇒ DP nhớ `k − 1` cột gần nhất.
- `max(a, c)` trong chuyển trạng thái ⇒ tách hai miền, dùng prefix/suffix max để giảm một bậc.
- `n ≤ 100` gợi ý ngay `O(n³)`.

## 15. Biến thể của bài

1. Lưới **chữ nhật** `m × n` (chiều cao `∈ [0, m]`): công thức y hệt, `O(n · m²)`.
2. Hàng xóm **dọc** thay vì ngang: điểm trong cùng cột — bài trở nên tầm thường.
3. Mỗi cột có thao tác phải trả **phí** `k` ⇒ trừ `k` khi `h[j] > 0` (bài tập 5).
4. Tô từ **dưới lên**: lật lưới theo chiều dọc rồi giải như cũ.

In [20]:
# Biến thể 4: tô từ dưới lên = lật lưới
def score_bottom_up(grid, h):
    n = len(grid); total = 0
    for j in range(n):
        for i in range(n):
            white = i < n - h[j]
            nb_black = (j > 0 and i >= n - h[j - 1]) or (j + 1 < n and i >= n - h[j + 1])
            if white and nb_black:
                total += grid[i][j]
    return total

for _ in range(80):
    n = rng.randint(1, 4); g = [[rng.randint(0, 6) for _ in range(n)] for _ in range(n)]
    b = max(score_bottom_up(g, h) for h in itertools.product(range(n + 1), repeat=n))
    assert b == Solution().maximumScore(g[::-1])
print('Biến thể tô từ dưới lên = bài gốc trên lưới lật dọc: khớp brute.')

Biến thể tô từ dưới lên = bài gốc trên lưới lật dọc: khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi `h` | mũ | Oracle |
| 2 | DP `(h[j−1], h[j])` trực tiếp | `O(n⁴)` | Dễ viết, chậm |
| 3 | DP + suffix/prefix max | `O(n³)` | **Nộp bài** |
| 4 | Pick / skip | `O(n³)` | Bộ nhớ nhỏ, khó nghĩ ra hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `grid = [[1, 100], [1, 100]]`.

**Bài 2 (tính tay):** `grid = [[4, 0, 4], [0, 5, 0], [4, 0, 4]]`.

**Bài 3 (điền TODO):** trả về cả điểm tối ưu **và** vector `h` đạt được, chạy `O(n³)` (lưu vết trong DP cách 3).

**Bài 4 (tự viết lại):** biến thể 1 — lưới `m × n`.

**Bài 5 (mini challenge):** mỗi cột có `h[j] > 0` phải trả phí `k`; tối đa hoá `điểm − phí`.

In [21]:
def solve_with_heights_todo(grid):
    # TODO
    raise NotImplementedError

def rectangular_todo(grid):
    # TODO
    raise NotImplementedError

def with_fee_todo(grid, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tô cột nào để lộ cột 100? (2) Tô trọn cột 0 (mất `1 + 1` vốn cũng không được tính). (3) `200`.

**Bài 2.** (1) Tô cột giữa lộ `4+4` ở hai cột bên, nhưng mất ô `5`? (2) So với tô cột 0 và 2 tới hàng 2 để lộ trọn cột giữa (`5`). (3) Tô trọn cột giữa thắng: lộ cả bốn ô `4` ⇒ `16` (`h = [0, 3, 0]`), hơn hẳn phương án lộ ô `5`.

**Bài 3.** (1) Bên cạnh `suf[c]` lưu `argmax a`. (2) Bên cạnh `pre` lưu `argmax a`. (3) Lưu `par[j][b][c] = a`, truy ngược từ `(b*, 0)`.

**Bài 4.** (1) `S_j` dài `m + 1`. (2) Chiều cao `∈ [0, m]`. (3) Vòng lặp theo `n` cột, mỗi cột `O(m²)`.

**Bài 5.** (1) Trừ `k` vào `new[b][c]` khi `c > 0`, nhưng cẩn thận cột ảo cuối. (2) Đơn giản hơn: trừ `k` khi **chốt** `b > 0` của cột `j`. (3) Kiểm bằng brute có phí.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().maximumScore([[1, 100], [1, 100]]) == 200
g2 = [[4, 0, 4], [0, 5, 0], [4, 0, 4]]
ans2 = Solution().maximumScore(g2)
assert ans2 == brute(g2) == 16
print('Bài 1: 200 | Bài 2:', ans2, '| một cấu hình tối ưu:', dp_with_heights(g2)[1])

Bài 1: 200 | Bài 2: 16 | một cấu hình tối ưu: [0, 3, 0]


In [23]:
# Bài 3, 4, 5 — viết chung cho lưới m x n có phí k mỗi cột được tô
def general(grid, k=0, want_heights=False):
    m, n = len(grid), len(grid[0])
    if n == 1:
        return (0, [0]) if want_heights else 0
    S = [[0] * (m + 1) for _ in range(n)]
    for j in range(n):
        for i in range(m):
            S[j][i + 1] = S[j][i] + grid[i][j]
    H = m + 1
    dp = [[NEG] * H for _ in range(H)]
    dp[0] = [0 if b == 0 else -k for b in range(H)]          # trả phí khi chọn h[0] > 0
    par = []
    for j in range(n):
        Sj = S[j]
        new = [[NEG] * H for _ in range(H)]; pj = [[0] * H for _ in range(H)]
        for b in range(H):
            suf = [NEG] * (H + 1); sarg = [0] * (H + 1)
            for a in range(H - 1, -1, -1):
                v = dp[a][b] + (Sj[a] - Sj[b] if a > b else 0)
                if v > suf[a + 1]:
                    suf[a], sarg[a] = v, a
                else:
                    suf[a], sarg[a] = suf[a + 1], sarg[a + 1]
            pre, parg = NEG, 0
            for c in range(H):
                v, arg = suf[c], sarg[c]
                if pre > NEG:
                    w = pre + (Sj[c] - Sj[b] if c > b else 0)
                    if w > v:
                        v, arg = w, parg
                fee = k if (c > 0 and j < n - 1) else 0         # phí của cột j+1 (nếu là cột thật)
                new[b][c] = v - fee; pj[b][c] = arg
                if dp[c][b] > pre:
                    pre, parg = dp[c][b], c
        dp = new; par.append(pj)
    best_b = max(range(H), key=lambda b: dp[b][0])
    best = dp[best_b][0]
    if not want_heights:
        return best
    h = [0] * (n + 1); h[n] = 0; h[n - 1] = best_b
    for j in range(n - 1, 0, -1):
        h[j - 1] = par[j][h[j]][h[j + 1]]
    return best, h[:n]

def brute_general(grid, k=0):
    m, n = len(grid), len(grid[0]); best = NEG
    for h in itertools.product(range(m + 1), repeat=n):
        t = 0
        for j in range(n):
            for i in range(m):
                if i >= h[j] and ((j > 0 and h[j - 1] > i) or (j + 1 < n and h[j + 1] > i)):
                    t += grid[i][j]
        best = max(best, t - k * sum(1 for x in h if x > 0))
    return best

for _ in range(150):
    m = rng.randint(1, 4); n = rng.randint(1, 4); k = rng.choice([0, 0, 1, 3, 7])
    g = [[rng.randint(0, 6) for _ in range(n)] for _ in range(m)]
    assert general(g, k) == brute_general(g, k), (g, k)
    if k == 0 and m == n:
        assert general(g) == Solution().maximumScore(g)
    v, h = general(g, k, want_heights=True)
    S_score = sum(g[i][j] for j in range(n) for i in range(m)
                  if i >= h[j] and ((j > 0 and h[j - 1] > i) or (j + 1 < n and h[j + 1] > i)))
    assert v == S_score - k * sum(1 for x in h if x > 0), (g, k, h)
print('Bài 3 (khôi phục h), Bài 4 (m x n), Bài 5 (phí k): khớp brute.')

Bài 3 (khôi phục h), Bài 4 (m x n), Bài 5 (phí k): khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp[a][b] (h[j-1]=a, h[j]=b); dp init: a = 0, mọi b = 0
for j: for b:
    suf[c] = max_{a>=c} dp[a][b] + gain(a, b)
    pre    = max_{a<c}  dp[a][b]
    new[b][c] = max(suf[c], pre + gain(c, b))
ans = max_b dp[b][0]
gain(x, b) = S_j[x] - S_j[b] if x > b else 0
```

**Checklist:**

- [ ] Chiều cao chạy `0..n` (có cả 0)?
- [ ] Ô trắng có **cả hai** hàng xóm đen chỉ tính **một** lần (dùng `max(L, R)`)?
- [ ] Trừ phần bị tô đen `S_j(h[j])`?
- [ ] Cột ảo hai biên chiều cao 0; `n = 1` trả 0?
- [ ] Độ phức tạp `O(n³)` (không để `O(n⁴)` với `n = 100`)?